## 1. Introduction

<div style="padding: 5px; border-left: 5px solid #ceb431; margin-bottom: 10px;">
    <strong>October 27th, 2026</strong>
</div>

In notebooks 01 and 02, I documented the process of becoming familiar with the fundamentals of quantum mechanics, primarily for the purpose of gaining a firm rooted understanding of the quantum-related field before attempting to experiment with quantum machine learning (QML).

To get to a point where I feel comfortable implementing qubits into a PyTorch sequence, I will first have to get comfortable simulating qubits in Python at all.

A lot of the mental modeling surrounding quantum fundamentals has already been centered around simulating it in Python, as I've had that idea from the beginning. Through this and a bit research, I've come to using a class to construct quantum circuits, and doing operations with the highest level of abstraction being the circuit object.

In [333]:
import numpy as np

## 2. Understanding the Planned Structure and Concepts

So the general idea I've landed on is having a QuantumCircuit class with gates as the methods. 

My initial thoughts of how to approach the problem was creating a Qubit class and having each qubit object simply store its own state vector. This initially seemed like a solid way to store qubits and their values, but I quickly hit a wall that made the whole idea seem silly. In operating on your circuit, you're inevitably shattering the individual qubits' state vectors and identity and merging them into one quantum circuit state vector. So, it became rather obvious that the issues that would arise from trying to translate qubit objects into circuits would be too much of a headache. From there, I searched for ideas on how to handle it before finding out that it would be most efficient to simply store circuits as objects referencing a big array of states. It was really challenging to understand that the state vector for a large circuit could be cleanly represented in code as a single dimensional array, so it took a lot of reframing to realize that this was a solid path.

#### Class Skeleton and Initial File Write

In [334]:
%%writefile quantum_circuit.py

class QuantumCircuit:
    pass

Overwriting quantum_circuit.py


This is going to introduce a few concepts that I have never had to deal with in my independent projects. So, first I am going to get an understanding of some of that stuff and how they'll fit in to the final reusable form of the code.

The first one of these core concepts to understand is going to be np arrays containing complex numbers.

#### Complex Numbers in numpy

I didn't know this before starting on this section but Python natively supports complex numbers using just the letter j. 

j in python represents $\sqrt{-1}$, whats known to me as i.

Using this is as simple as following any number immediately(no space) with j, such as: 2j, 4.56j, 1j, 22j etc.

In [335]:
my_number = 1j

print(my_number, my_number**2)

my_number = my_number**2

print(my_number)

1j (-1+0j)
(-1+0j)


Interesting that it keeps the j there, despite being 0.

So, to create an array of complex numbers in numpy, we can just do:

In [336]:
my_list = np.array([244, 2, 1, 18.6], dtype=complex)
print(my_list)

#or 

my_list = np.array([244, 2, 1, 18.6], dtype=np.complex128)
print(my_list)

#or

my_list = np.array([244, 2, 1, 18.6])
my_list = np.complex128(my_list)
print(my_list)

#or 

my_list = np.array([244, 2, 1, 18.6])
my_list = my_list.astype(complex)
print(my_list)


[244. +0.j   2. +0.j   1. +0.j  18.6+0.j]
[244. +0.j   2. +0.j   1. +0.j  18.6+0.j]
[244. +0.j   2. +0.j   1. +0.j  18.6+0.j]
[244. +0.j   2. +0.j   1. +0.j  18.6+0.j]


And we can select/modify the real and imaginary part of the complex number with:

In [337]:
my_list = np.array([244, 2, 1, 18.6], dtype=complex)

print(my_list.real, my_list.imag)

my_list.imag = np.array([1, 2, 3, 4])

print(my_list.real, my_list.imag)

print(my_list)


[244.    2.    1.   18.6] [0. 0. 0. 0.]
[244.    2.    1.   18.6] [1. 2. 3. 4.]
[244. +1.j   2. +2.j   1. +3.j  18.6+4.j]


Nice.

Now we can represent a single qubit's state vector with a numpy array:

In [338]:
my_qubit_sv = np.complex128([1.0, 0.0])

print(my_qubit_sv)

[1.+0.j 0.+0.j]


Where 
my_qubit_sv's indices represent each possibility in the state vector 
and
my_qubit_sv[i] represents a probability amplitude

So, currently: my_qubit_sv = $1.0|0\rangle$, $0.0|1\rangle$

To apply a H-Gate to that qubit, we just need to multiply it's probability amplitudes by the H-gate matrix:

$H = \begin{pmatrix} \frac{1}{\sqrt{2}}, \frac{1}{\sqrt{2}} \\ \frac{1}{\sqrt{2}}, -\frac{1}{\sqrt{2}} \end{pmatrix}$

In [339]:
example_h_matrix = np.array([[1/np.sqrt(2), 1/np.sqrt(2)],[1/np.sqrt(2), -(1/np.sqrt(2))]], dtype=np.complex128)

my_qubit_sv @ example_h_matrix 


array([0.70710678+0.j, 0.70710678+0.j])

However, that only works for a single qubit, if we scale it up to be more than one:

In [340]:
example_circuit = np.complex128([1.0, 0.0, 0.0, 0.0])
#Represents a 2 qubit circuit in North Pole position (100% chance to read 00)

#example_circuit @ example_h_matrix 
## Will cause a dot product dimensional error

This is true beyond our simulation, you cannot apply a 2x2 matrix to a 2 qubit circuit, you must first scale your matrix up by taking the tensor product of the H-matrix and the Identity matrix. I haven't yet discussed the identity matrix in these notebooks, but they've been covered extensively in my linear algebra class, so I'm quite aware of their basic properties.

<div style="padding: 5px; border-left: 5px solid #ceb431; margin-bottom: 10px;">
    <strong>October 28th, 2026</strong>
</div>

I realized that I wasn't quite sure how to implement a tensor product function into my code. Looking into it, its conceptually just "for every element in Matrix A, multiply it by the whole matrix B and combine the resulting matrices into a larger matrix.

So, for:

$A = \begin{pmatrix}a, b \\ c, d \end{pmatrix}$ 
and 
$B = \begin{pmatrix}w, x\\ y, z\end{pmatrix}$

The tensor product would be:

$A \otimes B = \begin{pmatrix}a \begin{pmatrix}w, x \\ y, z\end{pmatrix}, b \begin{pmatrix}w, x \\ y, z\end{pmatrix} \\ c \begin{pmatrix}w, x \\ y, z\end{pmatrix}, d \begin{pmatrix}w, x \\ y, z\end{pmatrix} \end{pmatrix}$

Which turns into:

$A \otimes B = \begin{pmatrix} aw, ax, bw, bx, \\ ay, az, by, bz \\ cw, cx, dw, dx \\ cy, cz, dy, dz\end{pmatrix}$

#### Tensor product of H and the identity matrix to allow for us to apply an H-gate to a sinlge qubit in a dual qubit circuit:

##### In LaTex:

for:

$H = \begin{pmatrix} \frac{1}{\sqrt{2}}, \frac{1}{\sqrt{2}} \\ \frac{1}{\sqrt{2}}, -\frac{1}{\sqrt{2}} \end{pmatrix}$

and

$I = \begin{pmatrix} 1, 0 \\ 0, 1 \end{pmatrix}$

Tensor product:

$H \otimes I = \begin{pmatrix}\frac{1}{\sqrt{2}} \begin{pmatrix}1, 0 \\ 0, 1\end{pmatrix}, \frac{1}{\sqrt{2}} \begin{pmatrix}1, 0 \\ 0, 1\end{pmatrix} \\ \frac{1}{\sqrt{2}} \begin{pmatrix}1, 0 \\ 0, 1\end{pmatrix}, -\frac{1}{\sqrt{2}} \begin{pmatrix}1, 0 \\ 0, 1\end{pmatrix} \end{pmatrix}$

Which turns into:

$H \otimes I = \begin{pmatrix} \frac{1}{\sqrt{2}}, 0, \frac{1}{\sqrt{2}}, 0, \\ 0, \frac{1}{\sqrt{2}}, 0, \frac{1}{\sqrt{2}} \\ \frac{1}{\sqrt{2}}, 0, -\frac{1}{\sqrt{2}}, 0 \\ 0, \frac{1}{\sqrt{2}}, 0, -\frac{1}{\sqrt{2}}\end{pmatrix}$

##### In Python:

In [341]:
example_identity_matrix = np.complex128([
    [1, 0],
    [0, 1]
])

#HxI Tensor product matrix mult/merging
example_h_by_identity = example_identity_matrix*example_h_matrix[0][0]
example_h_by_identity = np.hstack((example_h_by_identity, example_identity_matrix*example_h_matrix[0][1]))
example_h_by_identity = np.vstack(
    (example_h_by_identity,  np.hstack(
        (example_identity_matrix*example_h_matrix[1][0], example_identity_matrix*example_h_matrix[1][1])
        )))

example_hbi_superpos_state = example_h_by_identity @ example_circuit
example_hbi_superpos_state


array([0.70710678+0.j, 0.        +0.j, 0.70710678+0.j, 0.        +0.j])

Perfect! this represnts a quantum circuit percentage-based state vector of 50% $|00\rangle$ + 50%$|01\rangle$ + 0% $|10\rangle$ + 0% $|11\rangle$

This is aligned with what we'd expect by applying an H-gate to one qubit in a dual qubit circuit. By putting qubit 1 into superposition, pointing to the equator on the bloch sphere, we have made it equally likely for qubit 1 to measure 1 or 0. Therefore, it will be either 00 or 01, with a 50% chance to be each.

To convert the complex number probability amplitude to the percentage, you just take the absolute value of the amplitude squared:

In [342]:
example_hbi_prob_amps_in_percent = abs(example_hbi_superpos_state**2)
example_hbi_prob_amps_in_percent

array([0.5, 0. , 0.5, 0. ])

#### Tensor product of $H \otimes H$ 

While figuring that out, I noticed that applying an H-gate to both qubits is possible by multiplying the state vector by $H \otimes H$ instead of $H \otimes I$, like so:

In [343]:
#HxH Tensor product matrix mult/merging
example_h_by_h = example_h_matrix*example_h_matrix[0][0]
example_h_by_h = np.hstack((example_h_by_h, example_h_matrix*example_h_matrix[0][1]))
example_h_by_h = np.vstack(
    (example_h_by_h,  np.hstack(
        (example_h_matrix*example_h_matrix[1][0], example_h_matrix*example_h_matrix[1][1])
        )))

example_hbh_superpos_state = example_h_by_h @ example_circuit

example_hbh_prob_amps_in_percent = abs(example_hbh_superpos_state**2)
example_hbh_prob_amps_in_percent


array([0.25, 0.25, 0.25, 0.25])

Awesome! both qubits being put into equator superposition would result in exactly what we see here, an equal chance of every result when measured. Our np array represents a percentage-based state vector of:

$\psi$ = 25% $|00\rangle$ + 25%$|01\rangle$ + 25% $|10\rangle$ + 25% $|11\rangle$

For the final implementation, these tensor products are going to be massive. Therefore, I think it'd be good to spend some time workshopping a way to implement this with computational efficiency.

#### Understanding Binary Indices

I am aware from my research rabbit holes that the best method is going to be using the actual bitwise representation of the indices as the possibilites. For example, in a np array that is supposed to represent:

$\psi$ =  $|00\rangle$ + $|01\rangle$ + $|10\rangle$ + $|11\rangle$

The np array would look like: 

array([w, x, y, z])

And the indices of that array would be 0, 1, 2, and 3.

if you convert each of the index values themselves into binary with python, you would get a few different formats depending on the method, with one of them standing out as a good representation of state possibilities:

In [344]:
print(bin(0), bin(1), bin(2), bin(3))

print(f"{0:b} {1:b} {2:b} {3:b}")

print(f"{0:02b} {1:02b} {2:02b} {3:02b}") #number before b is amount of padding and other is spacing

0b0 0b1 0b10 0b11
0 1 10 11
00 01 10 11


As long as the amount of padding variable remains equal to the number of neurons, each index value perfectly matches up with a corresponding possible state. As I said before, I cannot take credit for this idea, but it is brilliantly efficient for this use case. 

To be able to use this information properly, I must run some tests to make sure it lines up properly, then I can attempt to form efficient operations utilizing it.

As we saw in the previous output, the binary output lines up with a circuit's possible states, such as:

$\psi$ =  $|00\rangle$ + $|01\rangle$ + $|10\rangle$ + $|11\rangle$

But i'll have to make sure the indexes and their values actually line up with the exact outputs of my scripts. The first step towards that, to me, is making sure that the values we get after applying a gate line up with the states automatically.

Let's say we initialized a dual qubit circuit, assuming default starting positions gives us a state vector of:

$\psi$ =  1.0 $|00\rangle$ + 0.0 $|01\rangle$ + 0.0 $|10\rangle$ + 0.0 $|11\rangle$

because both qubits are independent and will always measure to 0 when measured.

In python, this is represented as:

In [345]:
example_circuit_it = np.complex128([1.0, 0.0, 0.0, 0.0])
example_circuit_it

array([1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j])

Right now, our binary representation of each index value perfectly lines up. We can test this by doing:

In [346]:
def example_show_bin(ex_qin, ex_circ_in):
    ex_padding = ex_qin # for number of qubits in the circuit
    for i in range(len(ex_circ_in)):
        print(f"{ex_circ_in[i]}: {i:0{ex_padding}b}")
example_show_bin(2, example_circuit_it)

(1+0j): 00
0j: 01
0j: 10
0j: 11


So far so good, now lets make sure it holds true after an H-gate is applied to just qubit 1:

Applying an H-gate to the first neuron would result in the following percentage-based state vector:

$\psi$ =  50% $|00\rangle$ + 0% $|01\rangle$ + 50% $|10\rangle$ + 0% $|11\rangle$

In [347]:
example_show_bin(2, example_hbi_superpos_state)

print(f"\nPercentages: {abs(example_hbi_superpos_state**2)}")

(0.7071067811865475+0j): 00
0j: 01
(0.7071067811865475+0j): 10
0j: 11

Percentages: [0.5 0.  0.5 0. ]


Perfect, thats enough to make me confident enough to at least start designing an initialization method to ease further testing.

#### Creating the Init Method

When initializing a quantum circuit object, you'd obviously need to pass in the number of qubits, and we'll have it start in the standard North-Pole position:

In [348]:
%%writefile quantum_circuit_nb03.py
import numpy as np

Overwriting quantum_circuit_nb03.py


In [349]:
%%writefile -a quantum_circuit_nb03.py

class QuantumCircuit:
    def __init__(self, n_qubits):
        self.n_qubits = n_qubits
        self.size = 2**n_qubits
        self.state_vector = np.zeros(2**n_qubits, dtype=np.complex128)
        self.state_vector[0] = 1.0

Appending to quantum_circuit_nb03.py


In [350]:
%run -i quantum_circuit_nb03.py

three_q_circ = QuantumCircuit(3)

print(three_q_circ.state_vector)

[1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]


We now have a class that allows us to initialize a quantum circuit with $2^n$ probability amplitudes, initialized properly to North-Pole position.

#### Creating an Index Value Debugging Method

Let's add a little method for showing the binarized index of each amplitude for easy viewing on small circuits and debugging:

In [351]:
%%writefile -a quantum_circuit_nb03.py

    def show_amp_bin(self):
        padding = self.n_qubits
        for i, amplitude in enumerate(self.state_vector):
            print(f"{amplitude.real}: {i:0{padding}b}")

Appending to quantum_circuit_nb03.py


In [352]:
%run -i quantum_circuit_nb03.py

three_q_circ_bin = QuantumCircuit(3)

three_q_circ_bin.show_amp_bin()

1.0: 000
0.0: 001
0.0: 010
0.0: 011
0.0: 100
0.0: 101
0.0: 110
0.0: 111


#### Creating an H-gate Method

In order to calculate the affects of applying an h-gate onto a qubit while maintaining our current structure, we should look at exactly what happened in the np arrays we tested on earlier.

for example, earlier we got:
```
(0.7071067811865475+0j): 00
0j: 01
(0.7071067811865475+0j): 10
0j: 11
Percentages: [0.5 0.  0.5 0. ]
```

from applying an H-gate to the first qubit in a 2 qubit circuit (started as North-Pole). Im hoping to use that sequence and result to design a method to apply H-gates to qubits cleanly and efficiently.

Applying an H-gate to the first qubit caused the array to go from:

```
1.0+0j: 00
0j: 01
0j: 10
0j: 11
```

to:

```
(0.7071067811865475+0j): 00
0j: 01
(0.7071067811865475+0j): 10
0j: 11
```







I've done some sketching and I've come up with a first iteration of the H-gate method to test. The method will find the elements in the primary np array that are greater than 0 and give half their amplitude value to the other possibility associated with flipping that bit.

For example, calling the h(1) method on this state vector (argument is the targeted bit):

1.0: 00
0  : 01
0  : 10
0  : 11

The program would iterate through until it immediately finds a non-zero value for possibility 00, half its value, and give the other half to "10" because that is the equivalent of making it a 50/50 chance of the targeted bit being 1 or 0. 

0.5: 00
0  : 01
0.5: 10
0  : 11

At first I didn't think that this would scale, but a few edge tests passed, such as applying an H-gate one after another. Applying a second H-gate to the exact same bit would result in nothing happening, as it would take half of both and give the other possibility the same amount.

With that being said, lets try to implement this and run some tests:

**Note: I later realized that dividing by 2 is how you'd do it if you were dealing purely with percentages. By dividing by 2 and later squaring it for the actual percentage, they won't add up to 100% or function correctly. I have made sure to implement the correct multiplication by 1/np.sqrt(2) for the H-gate implementation, but have decided to leave the original thought process in the notes.**

So effectively what we want is to find a value in the array that is not 0, then take that binary index, say: ```010``` (target 1) and give half of it's value to the binary index where the target bit is flipped: ```000```. Seems easy enough

With a google search, I found that people actually use something called a mask for this? In our case, all we have to do is specify the index of the target and do a bitwise operator:

In [353]:
my_ex_target = 2

mask = 1 << 2
result = 6 & mask
print(result)

print(f"{my_ex_target:08b}") # 10
print(f"{mask:08b}") # 100 (left shifted once)
print(f"{100:08b}, {100 ^ mask:08b}") 
print(f"{4:08b}, {4 ^ mask:08b}") 
print(f"{9:08b}, {9 ^ mask:08b}") 
print(f"{2:08b}, {2 ^ mask:08b}") 

#RETURNS THE EXACT SAME BINARY NUMBER BUT SWAPS THE BIT AT THE TARGETED INDEX!

4
00000010
00000100
01100100, 01100000
00000100, 00000000
00001001, 00001101
00000010, 00000110


So now we can rely on this switching the bit and calling it directly using that:

In [354]:
%%writefile -a quantum_circuit_nb03.py

    def h(self, target_idx):
        mask = 1 << target_idx
        h_mult = 1/np.sqrt(2)
        new_state = np.copy(self.state_vector)

        for i in range(self.size):
            if (i & mask) == 0:
                delta_i1 = (self.state_vector[i] + self.state_vector[i ^ mask])*h_mult
                delta_i2 = (self.state_vector[i] - self.state_vector[i ^ mask])*h_mult

                new_state[i] = delta_i1
                new_state[i ^ mask] = delta_i2

        self.state_vector = new_state
        
        return self

Appending to quantum_circuit_nb03.py


In [355]:
%run -i quantum_circuit_nb03.py

two_q_circ_h = QuantumCircuit(2)

print("BEFORE H-GATE:")
two_q_circ_h.show_amp_bin()

print("AFTER H-GATE:")
two_q_circ_h.h(0)

two_q_circ_h.show_amp_bin()

BEFORE H-GATE:
1.0: 00
0.0: 01
0.0: 10
0.0: 11
AFTER H-GATE:
0.7071067811865475: 00
0.7071067811865475: 01
0.0: 10
0.0: 11


Awesome, love to see that working, let's try a few test cases to make sure it's working properly:

In [356]:
#Applying another H-gate to the same target bit

print("BEFORE H-GATE:")
two_q_circ_h.show_amp_bin()

print("AFTER H-GATE:")
two_q_circ_h.h(0)

two_q_circ_h.show_amp_bin()

BEFORE H-GATE:
0.7071067811865475: 00
0.7071067811865475: 01
0.0: 10
0.0: 11
AFTER H-GATE:
0.9999999999999998: 00
0.0: 01
0.0: 10
0.0: 11


At first I was confused and ready to begin debugging, but I realized that it is working perfectly. The inverse of an H-gate is an H-gate! So we're seemingly all good there.

In [357]:
#Applying H-gate to the other target bit AND THEN the same one

print("BEFORE H-GATE:")
two_q_circ_h.show_amp_bin()

print("AFTER H-GATE:")
two_q_circ_h.h(1).h(0) # I found out that if you return self you can make the gates chain-able like this!

two_q_circ_h.show_amp_bin()

print("\n", two_q_circ_h.state_vector.real**2) # checking with percentages

BEFORE H-GATE:
0.9999999999999998: 00
0.0: 01
0.0: 10
0.0: 11
AFTER H-GATE:
0.49999999999999983: 00
0.49999999999999983: 01
0.49999999999999983: 10
0.49999999999999983: 11

 [0.25 0.25 0.25 0.25]


Perfect! let's do one last test on a bigger circuit:

In [358]:
three_q_circ_h = QuantumCircuit(3)

print("BEFORE H-GATE:")
three_q_circ_h.show_amp_bin()

print("AFTER H-GATE:")
three_q_circ_h.h(2).h(1)

three_q_circ_h.show_amp_bin()

print("\n", three_q_circ_h.state_vector.real**2)

BEFORE H-GATE:
1.0: 000
0.0: 001
0.0: 010
0.0: 011
0.0: 100
0.0: 101
0.0: 110
0.0: 111
AFTER H-GATE:
0.4999999999999999: 000
0.0: 001
0.4999999999999999: 010
0.0: 011
0.4999999999999999: 100
0.0: 101
0.4999999999999999: 110
0.0: 111

 [0.25 0.   0.25 0.   0.25 0.   0.25 0.  ]


I am confident enough in this H-gate implementation to continue developing the next section.